# Advanced Prompt Engineering — AI Engineering Interview Prep

Chain-of-Thought (CoT), ReAct, and self-consistency techniques with Claude API.

**Prerequisites**: Set `ANTHROPIC_API_KEY` environment variable.

In [ ]:
import anthropic
import os
import json
import re
from collections import Counter

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-haiku-4-5-20251001"

def ask(prompt, system=None, max_tokens=500, temperature=1.0):
    kwargs = {
        "model": MODEL, "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}]
    }
    if system:
        kwargs["system"] = system
    resp = client.messages.create(**kwargs)
    return resp.content[0].text

print(f"Model: {MODEL}")

## 1. Chain-of-Thought (CoT) Prompting

In [ ]:
problem = """A model trained on 10,000 examples achieves 95% training accuracy and 72% validation accuracy.
After adding dropout (p=0.5) and L2 regularization (lambda=0.01), training accuracy drops to 88%
and validation accuracy improves to 84%. The team wants to deploy but is concerned.
Should they deploy? What should they do next?"""

# Without CoT
direct = ask(f"Answer this ML question:\n{problem}")
print("=== Direct (no CoT) ===")
print(direct)

# With CoT - 'think step by step'
cot_prompt = f"""Answer this ML question. Think step by step before giving your final recommendation.

{problem}"""
cot = ask(cot_prompt, max_tokens=600)
print("\n=== Chain-of-Thought ===")
print(cot)

## 2. Few-Shot CoT

In [ ]:
few_shot_cot = """Diagnose ML problems by reasoning step by step.

Problem: Train loss=0.05, Val loss=3.2
Reasoning: Train loss is very low but val loss is much higher → large gap = overfitting.
The model memorized training data. Solutions: more data, dropout, regularization, simpler model.
Diagnosis: Severe overfitting.

Problem: Train loss=2.8, Val loss=2.9
Reasoning: Both losses are high and similar → no gap means model isn't overfitting.
High loss on both sets = underfitting. Model lacks capacity or hasn't trained enough.
Solutions: larger model, more epochs, lower learning rate, better features.
Diagnosis: Underfitting.

Problem: Train loss=0.15, Val loss=0.18
Reasoning:"""

result = ask(few_shot_cot)
print("Few-shot CoT diagnosis:")
print(result)

## 3. ReAct (Reason + Act) Pattern

In [ ]:
# ReAct: interleave Thought → Action → Observation cycles
# In a real system, Actions would call tools. Here we simulate the pattern.

react_system = """You are an ML debugging assistant using the ReAct framework.
For each step, output:
Thought: [your reasoning]
Action: [what you would check/do]
Observation: [what you hypothetically find]

After 3 cycles, give a Final Answer."""

react_problem = """A production ML model's accuracy dropped from 94% to 71% overnight.
No code changes were made. Debug this issue."""

result = ask(react_problem, system=react_system, max_tokens=700)
print("ReAct debugging trace:")
print(result)

## 4. Self-Consistency

In [ ]:
# Self-consistency: sample multiple reasoning paths, majority vote
question = """A dataset has 1000 samples: 900 class A and 100 class B.
A model predicts all samples as class A. What is the accuracy and F1 score?
Answer with just: Accuracy=X%, F1=Y.YY"""

# Sample N times (would normally use temperature>0 for diversity)
N = 3
answers = []
for i in range(N):
    cot = ask(f"Think step by step, then answer: {question}", max_tokens=300)
    # Extract final answer line
    lines = [l.strip() for l in cot.strip().split('\n') if l.strip()]
    answers.append(cot)
    print(f"Sample {i+1}:")
    print(cot[:300])
    print("---")

print("\nSelf-consistency: majority vote over", N, "samples")
print("(In production: parse final answer labels and pick most common)")

## 5. Prompt Chaining

In [ ]:
# Prompt chaining: output of one prompt feeds into next
raw_description = """We need to build a system that takes customer support emails, figures out
what product they're about, how urgent it is, and routes them to the right team.
Sometimes emails are in Spanish or French too."""

# Step 1: Extract requirements
step1_prompt = f"""Extract technical requirements from this description. Output a numbered list.

Description: {raw_description}"""

requirements = ask(step1_prompt, max_tokens=300)
print("Step 1 — Requirements:")
print(requirements)

# Step 2: Map to ML components
step2_prompt = f"""For each requirement below, identify the ML component needed (classifier, NER, etc.).
Output as: Requirement → ML Component → Suggested model.

Requirements:
{requirements}"""

components = ask(step2_prompt, max_tokens=400)
print("\nStep 2 — ML Architecture:")
print(components)

# Step 3: Implementation plan
step3_prompt = f"""Given these ML components, create a 5-step implementation roadmap with time estimates.

{components}"""

roadmap = ask(step3_prompt, max_tokens=400)
print("\nStep 3 — Roadmap:")
print(roadmap)

## Interview Tips

- **CoT trigger**: 'think step by step' or 'let's work through this' unlocks reasoning in LLMs.
- **When CoT helps**: math, logic, multi-step reasoning. Less needed for simple retrieval tasks.
- **ReAct vs pure CoT**: ReAct is CoT + actions (tool calls). Use ReAct when model needs external info.
- **Self-consistency cost**: N completions = N× the cost. Use for high-stakes decisions.
- **Prompt chaining**: decompose complex tasks; each step's output is structured input for the next.
- **Zero-shot CoT**: just appending 'Think step by step' to a prompt — surprisingly effective.